#📚 文系大学生による『実践データ分析100本ノック＆LLM活用20本ノック』学習ログ
**著： 岡野 菜月（四国大学文学部 / 28卒）**

**GitHub： https://github.com/NatsukiOkano**

##📝 このノートブックの概要
＊ 書籍「Python 実践データ分析100本ノック 第3版」（秀和システム新社刊）の学習過程を記録したものです。

＊ 最大の特徴は、「文学部の視点でIT・データ分析を再解釈する」という試みにあります。

＊ 非情報系の学習者が直面する「専門用語の壁」を取り払うため、すべての処理に対し、論理的かつ直感的な「完全独自解説」を記述しています。

##⚠️ 注意事項
- **著作権保護の観点から、データセットおよび実行結果は同梱しておりません。**

### 教材
下山輝昌・松田雄馬・三木孝行 著「Python 実践データ分析100本ノック 第3版」（秀和システム新社、2025）

# 5章 顧客の退会を予測する１０本ノック

引き続き、スポーツジムの会員データを使って顧客の行動を分析していきます。  
３章では顧客の全体像を把握し、4章では数ヶ月利用している顧客の来月の利用回数の予測を行いました。   
ここでは、教師あり学習の分類を用いて、顧客の退会予測を取り扱います。

### ノック41：データを読み込んで利用データを整形しよう

In [ ]:
import pandas as pd
uselog_months = pd.read_csv('use_log_months.csv')
uselog_months.head()

In [ ]:
customer = pd.read_csv('customer_join.csv')
customer.head()

In [ ]:
year_months = list(uselog_months['年月'].unique())
uselog = pd.DataFrame()

for i in range(1, len(year_months)):
    tmp = uselog_months.loc[uselog_months['年月']==year_months[i]].copy()
    tmp.rename(columns={'count':'count_0'}, inplace=True)
    tmp_before = uselog_months.loc[uselog_months['年月']==year_months[i-1]].copy()
    del tmp_before['年月']
    tmp_before.rename(columns={'count':'count_1'}, inplace=True)
    tmp = pd.merge(tmp, tmp_before, on='customer_id', how='left')
    uselog = pd.concat([uselog, tmp], ignore_index=True)
uselog.head()

#list()：抽出したデータをPython標準の「リスト型」に変換する。
#df['列名'].unique：重複を削ぎ落した「純粋な種類の一覧」を1行で表示させる。
#pd.DataFrame：データフレーム(表)を作成する。

#for i in range：「繰り返し処理」の基本セット
  # for：～の間、繰り返す。
  # i：今、何回目か数えるための変数。
  # in range()：繰り返す回数。

#len：データの件数を数える。Excelのcount関数と同じ役割。
#tmp：変数
#tmp_before：変数

#df.loc[行ラベル,列ラベル]：ラベルで指定。ラベルを使って条件に合うデータを探す。
#df.iloc[]：整数で指定。整数を使って条件に合うデータを探す。

#df.copy()：複製。コピーして新しいデータフレーム(表)を作る。

#pd.concat()：結合。同じ形式(列名)のデータフレーム(表)を、縦(行方向)に結合する。
#ignore_index：0から順番に行番号(index)を新しく振り直す。
#ignore_index=True：元の行番号(index)を削除して、0から順番に行番号(index)を新しく振り直す。
#ignore_index=False：0から順番に行番号(index)を新しく振り直さず、元の行番号(index)を使う。

#pd.merge()：結合。正規化されてバラバラになったデータを、分析しやすいように「逆正規化（合体）」させる。
#pd.merge(左の表,右の表, left_on=, right_on=, how=)
#left_on：左側の表(テーブル)の使用する列名を指定。
#right_on：右側の表(テーブル)の使用する列名を指定。
#※left_on, right_on はそれぞれの表(テーブル)で使う列名が異なるときに使う。
#how：表(テーブル)の結合方法を指定。（※よく使う結合方法↓）
#how='left'：左の表を主役にする。右に一致するデータがなくても、左のデータはすべて残す。
#how='right'：右の表を主役にする。左に一致するデータがなくても、右のデータはすべて残す。
#how='inner'：両方の表にあるデータだけを残す。どちらか片方にしかないデータは消える。
#how='outer'：両方の表のデータをすべて残す。一致しない部分は「欠損値(NaN)」で埋める。

### ノック42：退会前月の退会顧客データを作成しよう

In [ ]:
from dateutil.relativedelta import relativedelta
exit_customer = customer.loc[customer['is_deleted']==1].copy()
exit_customer['exit_date'] = None
exit_customer['end_date'] = pd.to_datetime(exit_customer['end_date'])
for i in exit_customer.index:
    exit_customer.loc[i, 'exit_date'] = exit_customer.loc[i,'end_date'] - relativedelta(months=1)
exit_customer['exit_date'] = pd.to_datetime(exit_customer['exit_date'])
exit_customer['年月'] = exit_customer['exit_date'].dt.strftime('%Y%m')
uselog['年月'] = uselog['年月'].astype(str)
exit_uselog = pd.merge(uselog, exit_customer, on=['customer_id', '年月'], how='left')
print(len(uselog))
exit_uselog.head()

#from dateutil.relativedelta：『ライブラリ（dateutil）』に格納される『モジュール（relativedelta）』を指定する。
#import relativedelta：『モジュール（relativedelta）』に格納される『クラス（relativedelta）』を自分のプログラムに導入する。

#ライブラリ（dateutil）：日付や時刻の計算に特化したライブラリ。
#モジュール（relativedelta）：日付計算プログラムを格納したファイル。
#クラス（relativedelta）：日付計算AI。実際に日付計算を行う正体。

#「~」：否定演算子。「NOT（～ではない）」を意味する記号。※flg_is_null：変数。~flg_is_null：flg_is_nullの反対。
#「=」：代入演算子。代入。例）x = 100：xに100を代入。→これ以降、xは100として扱われる。
#「==」：比較演算子。比較。(判定)　例）x == 100：xの中身は100と同じ？→もしxが100ならTrueになる。

#pd.to_datetime：「文字(object)」として読み込まれた日付を、計算や加工ができる『日付専用データ(datetime)』に作り変える。
#to：～へ変換する。
#pd.to_datetimeの引数「format」：AIに日付(文字)の読み方を「%」で学習させる。
#日付(文字)の読み方：「%Y」西暦(4桁),「%y」西暦(2桁),「%m」月(2桁),「%d」日(2桁),「%H」時(2桁),「%M」分(2桁)

#df.astype('クラス')：クラス(データ型)を強制的に変更する。
#【astype()でよく使うクラス↓】
#  int：整数
#  float：少数
#  object：文字（pandasが、列の種類を表示するときに使う名前。）
#  ※str：文字列（Python(言語そのもの)が呼ぶ名前。）str = object

In [ ]:
exit_uselog = exit_uselog.dropna(subset=['name'])
print(len(exit_uselog))
print(len(exit_uselog['customer_id'].unique()))
exit_uselog.head()

#df.dropna()：全列が対象。1つでも欠損値(NaN)セルがあればその行ごと削除する。
#df.dropna(subset=['列名'])：指定した列だけが対象。欠損値(NaN)セルがあればその行ごと削除する。

#len：データの件数を数える。Excelのcount関数と同じ役割。
#df['列名'].unique：重複を削ぎ落した「純粋な種類の一覧」を1行で表示させる。

### ノック43：継続顧客のデータを作成しよう

In [ ]:
uselog.head()

In [ ]:
conti_customer = customer.loc[customer['is_deleted']==0]
conti_customer.head()

In [ ]:
conti_uselog = pd.merge(uselog, conti_customer, on=['customer_id'], how='left')
conti_uselog.head()

In [ ]:
print(len(conti_uselog))
conti_uselog = conti_uselog.dropna(subset=['name'])
print(len(conti_uselog))

In [ ]:
conti_uselog = conti_uselog.sample(frac=1,random_state=0).reset_index(drop=True)
conti_uselog = conti_uselog.drop_duplicates(subset='customer_id')
print(len(conti_uselog))
conti_uselog.head()

#df.sample()：データをシャッフルして、公平に選び出す。
#frac：割合
#sample(frac=1)：全データ(100%)をシャッフルして、公平に選び出す。

#reset_index(drop=True)：古い行番号(index)を削除して、「0から始まる新しい行番号(index)」を振り直す。
#reset_index(drop=False)：古い行番号(index)を「列(ラベル：index)」として残しつつ、「0から始まる新しい行番号(index)」を振り直す。

#drop_duplicates()：重複したデータを削除する。
#subset：実行する範囲の指定。

In [ ]:
predict_data = pd.concat([conti_uselog, exit_uselog], ignore_index=True)
print(len(predict_data))
predict_data.head()

#pd.concat()：結合。同じ形式(列名)のデータフレーム(表)を、縦(行方向)に結合する。
#ignore_index：0から順番に行番号(index)を新しく振り直す。
#ignore_index=True：元の行番号(index)を削除して、0から順番に行番号(index)を新しく振り直す。
#ignore_index=False：0から順番に行番号(index)を新しく振り直さず、元の行番号(index)を使う。

In [ ]:
#FutureWarning：将来のコード廃止・変更予告。
#「Warning(警告)」は「Error(廃止)」ではないため、基本無視して進めよう。

### ノック44：予測する月の在籍期間を作成しよう

In [ ]:
predict_data['period'] = 0
predict_data['now_date'] = pd.to_datetime(predict_data['年月'], format='%Y%m')
predict_data['start_date'] = pd.to_datetime(predict_data['start_date'])
predict_data.head()

#pd.to_datetime：「文字(object)」として読み込まれた日付を、計算や加工ができる『日付専用データ(datetime)』に作り変える。
#to：～へ変換する。
#pd.to_datetimeの引数「format」：AIに日付(文字)の読み方を「%」で学習させる。
#日付(文字)の読み方：「%Y」西暦(4桁),「%y」西暦(2桁),「%m」月(2桁),「%d」日(2桁),「%H」時(2桁),「%M」分(2桁)

In [ ]:
for i in range(len(predict_data)):
    delta = relativedelta(predict_data.loc[i, 'now_date'], predict_data.loc[i, 'start_date'])
    predict_data.loc[i, 'period'] = int(delta.years*12 + delta.months)
print(delta)
predict_data.head()

#クラス（relativedelta）：日付計算AI。実際に日付計算を行う正体。

### ノック45：欠損値を除去しよう

In [ ]:
#機械学習や深層学習など、ほとんど全てのAIは、欠損値があると対応できない。

In [ ]:
predict_data.isnull().sum()

In [ ]:
predict_data = predict_data.dropna(subset=['count_1'])
predict_data.isna().sum()

#isna() = isnull()　※「欠損値(NaN)セル」があるかチェックする。

### ノック46：文字列型の変数を処理できるように整形しよう

In [ ]:
#カテゴリカル変数：カテゴリー変数。データの「区分」を表す。
#ダミー変数化：フラグ化。文字列や複数の選択肢を、[True(1)」か「False(0)」だけで表す複数の列に作り替える。
#説明変数(X)：予測に使う変数
#目的変数(y)：予測したい変数
#離散値：数える。中間の値が存在しない。例）1人と2人の間はない。
#連続値：計測する。中間の値が無限に存在する。例）1cmと2cmの間はある。

In [ ]:
list(predict_data)

In [ ]:
target_col = ['campaign_name', 'class_name', 'gender', 'count_1', 'routine_flg', 'period', 'is_deleted']
predict_data = predict_data[target_col]
predict_data.head()

In [ ]:
predict_data = pd.get_dummies(predict_data, dtype=int)
predict_data.head()

#pd.get_dummies(df)：ダミー変数を作成する。

In [ ]:
del predict_data['campaign_name_通常']
del predict_data['class_name_ナイト']
del predict_data['gender_M']
predict_data.head()

#del：データそのものを抹消するためのPython標準の命令。

#複数の列をまとめて消す：uselog_months.drop(columns=['usedate','usesdate'], inplece=True)
#df.drop()：複数の「列(columns)」や「行(index)」をまとめて削除する。

### ノック47：決定木を用いて退会予測モデルを作成してみよう

In [ ]:
#分類：大量に読み込ませたデータを特定のクラスに振り分ける「教師あり学習」の手法。
#回帰：大量に読み込ませたデータから数値を予測する「教師あり学習」の手法。
#クラスタリング：大量に読み込ませたデータを類似性に基づいて自動的にグループ化する「教師なし学習」の手法。

#アルゴリズム：実行手順。計算手順。処理手順。

In [ ]:
from sklearn.tree import DecisionTreeClassifier
import sklearn.model_selection

#from sklearn.tree：『ライブラリ（sklearn）』に格納される『モジュール（tree）』を指定する。
#import DecisionTreeClassifier：『モジュール（tree）』に格納される『クラス（DecisionTreeClassifier）』を自分のプログラムに導入する。

#ライブラリ（sklearn）：「教師あり学習」「教師なし学習」に100%特化したライブラリ。※強化学習は非対応。
#モジュール（tree）：決定木のプログラムを格納したファイル。
#モジュール（model_selection）：AIの実力を測るプログラムを格納したファイル。
#クラス（DecisionTreeClassifier）：決定木AI。分類を担当。カテゴリカル変数(区分)に振り分ける。
#クラス（DecisionTreeRegressor）：決定木AI。回帰を担当。離散値や連続値(数値)を予測する。

#決定木AI：分類も回帰も両方できる、教師あり学習のAIモデル。
#決定木：最も綺麗にデータを「0と1」に分割できる「説明変数とその条件」を探す作業を木構造状に派生させていく作業。

In [ ]:
exit = predict_data.loc[predict_data['is_deleted']==1]
conti = predict_data.loc[predict_data['is_deleted']==0].sample(len(exit),random_state=0)

#df.loc[行ラベル,列ラベル]：ラベルで指定。ラベルを使って条件に合うデータを探す。
#df.iloc[]：整数で指定。整数を使って条件に合うデータを探す。

#df.sample()：データをシャッフルして、公平に選び出す。
#len：データの件数を数える。Excelのcount関数と同じ役割。

#random_state：AIのランダム性(気の変わり)を抑える。
#random_state=0：毎回、「0」が名前に設定されたデータを出力。

In [ ]:
X = pd.concat([exit, conti], ignore_index=True)
y = X['is_deleted']
del X['is_deleted']
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(X,y, random_state=0)

# X ：説明変数（予測に使う変数）
# y ：目的変数（予測したい変数）

#pd.concat()：結合。同じ形式(列名)のデータフレーム(表)を、縦(行方向)に結合する。
#ignore_index：0から順番に行番号(index)を新しく振り直す。
#ignore_index=True：元の行番号(index)を削除して、0から順番に行番号(index)を新しく振り直す。
#ignore_index=False：0から順番に行番号(index)を新しく振り直さず、元の行番号(index)を使う。

#train_test_split()：学習用データと評価用データに分割する。※過学習状態を避けるため。
#過学習状態：学習用データに過剰適応することで、未知なデータに対応できなくなること。
#※無指定の場合、学習用データ(75%)、評価用データ(25%)に分割される。

In [ ]:
model = DecisionTreeClassifier(random_state=0)
model.fit(X_train, y_train)
y_test_pred = model.predict(X_test)
print(y_test_pred)

#.fit：AIの学習を開始する。
# model.predict()：「学習済みAI」に「予測結果」を出してもらう。

In [ ]:
results_test = pd.DataFrame({'y_test':y_test, 'y_pred':y_test_pred})
results_test.head()

### ノック48：予測モデルの評価を行ない、モデルのチューニングをしてみよう

In [ ]:
#チューニング：AIが「最も高い精度」を出せる最高の設定を見つけ出す。
#乱数：「0～9」までの10種類の数字が、それぞれ同じ確率で現われるように並べられた数字の列。
#パラメータ：変数間を仲立ちする変数。

In [ ]:
correct = len(results_test.loc[results_test['y_test']==results_test['y_pred']])
data_count = len(results_test)
score_test = correct / data_count
print(score_test)

#len：データの件数を数える。Excelのcount関数と同じ役割。
#df.loc[行ラベル,列ラベル]：ラベルで指定。ラベルを使って条件に合うデータを探す。
#df.iloc[]：整数で指定。整数を使って条件に合うデータを探す。

In [ ]:
print(model.score(X_test, y_test))
print(model.score(X_train, y_train))

#model.score(説明変数,目的変数)：「モデル(AI)の予測精度(0～1)↓」を測る。
# 1 ：完璧。過学習状態を疑う。
# 0.7～0.9 ：優秀。ビジネス(商売)で自信をもって使えるレベル。
# 0.5～0.6 ：実用圏内。傾向はつかめている。もっとデータを増やせば予測精度が上がるかも。
# 0.3以下 ：厳しい。「説明変数が足りない」か、「データのバラつきが大きくて直線で表すには無理がある」。
# 0 ：使い物にならない。平均値を答えるより外れている。やり直し！

In [ ]:
X = pd.concat([exit, conti], ignore_index=True)
y = X['is_deleted']
del X['is_deleted']
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(X,y, random_state=0)

In [ ]:
model = DecisionTreeClassifier(random_state=0, max_depth=5)
model.fit(X_train, y_train)
print(model.score(X_test, y_test))
print(model.score(X_train, y_train))

#max_depth：「決定木の枝分かれを、最大で何段まで許可するか」を決める。
#max_depth=5：質問を5段階まで繰り返す。
#max_depth=None：制限なし。データが完全に分かれるまで、質問を繰り返す。

#None：値が存在しない。

### ノック49：モデルに寄与している変数を確認しよう

In [ ]:
importance = pd.DataFrame({'feature_naes':X.columns, 'coefficient':model.feature_importances_})
importance

#pd.DataFrame：データフレーム(表)を作成する。
#X.columns：変数名。データフレーム(表)の列ラベル。
#model.feature_importances_：ライブラリ（sklearn）で定義された決定木の係数(変数)。各説明変数の「予測への寄与度(重要度)」を「0～1」で表す。

In [ ]:
!pip install japanize_matplotlib
from sklearn import tree
import matplotlib.pyplot as plt
import japanize_matplotlib
%matplotlib inline

#!pip install：外部のライブラリをダウンロードする。
#※「!pip install」だけではプログラムで使えない。「import」する必要がある。

#ライブラリ（japanize_matplotlib）：「matplotlib」で作成したグラフを日本語化する。
#※「japanize_matplotlib」だけではグラフを作成できない。

#マジックコマンド（% , %%）：プログラムの外側の設定を変更する。
# %（ラインマジック）：その一行だけに効く命令。
# %%（セルマジック）：そのセル全体に効く命令。
#【よく使うマジックコマンド（↓）】
#%matplotlib inline（グラフをノートブック内に直接表示する。）
#%timeit（その行のコードの実行速度を計測する）
#%pwd（現在のフォルダにあるファイル一覧を表示する。）
#%run（外部のPythonファイルをノートブック上で実行する。）

In [ ]:
plt.figure(figsize=(20,8))
tree.plot_tree(model,feature_names=X.columns,fontsize=8)

#plt.figure()：グラフを描くための「白紙」を作成する。
#figsize=(横の長さ,縦の長さ)：plt.figure()の引数。グラフの大きさを指定する。
   #※単位：インチ(1インチ = 約2.54cm)

#plt.plot（折れ線グラフ）
#plt.bar（棒グラフ）
#plt.scatter（散布図）
#plt.hist（ヒストグラム）
#plot_tree（樹形図）

#plot_tree(学習済みAIモデル, feature_names=変数名, fontsize=)：樹形図
#model：変数。(model = DecisionTreeClassifier(random_state=0, max_depth=5))
#feature_names：分岐条件に表示させる変数名(列ラベル)。
#fontsize：文字の大きさ。

### ノック50：顧客の退会を予測しよう

In [ ]:
importance

In [ ]:
count_1 = 3
routine_flg = 1
period = 10
campaign_name = '入会費無料'
class_name = 'オールタイム'
gender = 'M'

In [ ]:
if campaign_name == '入会費半額':
    camaign_name_list = [1, 0]
elif campaign_name == '入会費無料':
    campaign_name_list = [0, 1]
elif campaign_name == '通常':
    campaign_name_list = [0, 0]
if class_name == 'オールタイム':
    class_name_list = [1, 0]
elif class_name == 'デイタイム':
    class_name_list = [0, 1]
if class_name == 'ナイト':
    class_name_list = [0, 0]
if gender == 'F':
    gender_list = [1]
elif gender == 'M':
    gender_list = [0]

#if：もし～なら
#else：そうでなければ
#elif：「else + if」あるいは～なら

#[1, 0],[0, 1],[0, 0],[1],[0]：ダミー変数。1(True), 0(False)
#[1, 0]：入会費半額
#[0, 1]：入会費無料
#[0, 0]：通常
#[1]：F(女性)
#[0]：M(男性)

#ダミー変数化する理由：文字を計算可能な数値として扱うため。
#ダミー変数 = 数値として扱うための「bool値」

In [ ]:
print(campaign_name_list)
print(class_name_list)
print(gender_list)

In [ ]:
input_data = [count_1, routine_flg, period]
input_data.extend(campaign_name_list)
input_data.extend(class_name_list)
input_data.extend(gender_list)
input_data = pd.DataFrame(data=[input_data], columns=X.columns)

#df.append()：指定したリストごと最後に追加する。
#df.extend()：指定したリストのデータのみ最後に追加する。
#【例：[1, 2, 3]に、[4, 5]を追加。↓】
# append() [1, 2, 3, [4, 5]]、extend() [1, 2, 3, 4, 5]

In [ ]:
print(model.predict(input_data))
print(model.predict_proba(input_data))

#model.predict(df)：AIで未来の数値を予測する。
#model.predict_proba(df)：分類AIが予測した区分が各カテゴリカル変数に該当する確率(0.0～1.0)を求める。

In [ ]:
#データドリブン：集めたデータを分析し、ビジネスの意思決定や企画立案、戦略策定などに役立てること。
#オーソドックス：標準的。
#プレ分析：自社の事業内容や目標を明確にし、分析の基盤を築くこと。業界全体の状況を把握し、競争環境や市場のトレンド(流行)を理解すること。
#相関分析：2つの変数の関係性を分析すること。